In [10]:
import json
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Load JSON file
with open("./iris_templates/BMT_(8x128)_templates.json", "r") as f:
    data1 = json.load(f)
    
with open("./iris_templates/iris_template_pooled_maxpool_mask.json", "r") as f:
    data2 = json.load(f)

In [11]:
import numpy as np

# === Utility Functions ===

def bitstring_to_array(bitstring):
    """Convert string of '0' and '1' to uint8 NumPy array."""
    return np.array([int(b) for b in bitstring], dtype=np.uint8)

def reshape_iris_code(flat_array, shape):
    """Reshape flat array to desired shape."""
    expected_size = np.prod(shape)
    if flat_array.size != expected_size:
        raise ValueError(f"Expected {expected_size} bits, got {flat_array.size}")
    return flat_array.reshape(shape)

# === Load and reshape templates ===

# === Parameters ===
TARGET_SHAPE = (2, 16, 256, 2)  # Change this as needed
EXPECTED_BITS = np.prod(TARGET_SHAPE)  # = 2 * 16 * 256 * 2 = 16384


samples_a = []
bad_entries = 0

for idx, entry in enumerate(data1):
    try:
        iris_template = entry.get("iris_template", {})
        iris_codes_raw = iris_template.get("iris_codes")
        mask_codes_raw = iris_template.get("mask_codes")

        if not iris_codes_raw or not mask_codes_raw:
            raise ValueError("Missing iris_codes or mask_codes")

        iris_codes = bitstring_to_array(iris_codes_raw)
        mask_codes = bitstring_to_array(mask_codes_raw)

        iris_codes = reshape_iris_code(iris_codes, TARGET_SHAPE)
        mask_codes = reshape_iris_code(mask_codes, TARGET_SHAPE)

        samples_a.append((iris_codes, mask_codes))

    except Exception as e:
        print(f"[Warning] Skipping entry {idx} due to error: {e}")
        samples_a.append(None)  # Keep index aligned
        bad_entries += 1

print(f"✅ Loaded {len(samples_a) - bad_entries} valid samples. Skipped {bad_entries} bad entries.")

# === Load and reshape templates ===

# === Parameters ===
TARGET_SHAPE = (2, 8, 128, 2)  # Change this as needed
EXPECTED_BITS = np.prod(TARGET_SHAPE)  # = 2 * 16 * 256 * 2 = 16384

samples_b = []
bad_entries = 0

for idx, entry in enumerate(data2):
    try:
        iris_template = entry.get("iris_template", {})
        iris_codes_raw = iris_template.get("iris_codes")
        mask_codes_raw = iris_template.get("mask_codes")

        if not iris_codes_raw or not mask_codes_raw:
            raise ValueError("Missing iris_codes or mask_codes")

        iris_codes = bitstring_to_array(iris_codes_raw)
        mask_codes = bitstring_to_array(mask_codes_raw)

        iris_codes = reshape_iris_code(iris_codes, TARGET_SHAPE)
        mask_codes = reshape_iris_code(mask_codes, TARGET_SHAPE)

        samples_b.append((iris_codes, mask_codes))

    except Exception as e:
        print(f"[Warning] Skipping entry {idx} due to error: {e}")
        samples_b.append(None)  # Keep index aligned
        bad_entries += 1

print(f"✅ Loaded {len(samples_b) - bad_entries} valid samples. Skipped {bad_entries} bad entries.")


[Warning] Skipping entry 0 due to error: 'NoneType' object has no attribute 'get'
[Warning] Skipping entry 1 due to error: 'NoneType' object has no attribute 'get'
[Warning] Skipping entry 2 due to error: 'NoneType' object has no attribute 'get'
[Warning] Skipping entry 3 due to error: 'NoneType' object has no attribute 'get'
[Warning] Skipping entry 4 due to error: 'NoneType' object has no attribute 'get'
[Warning] Skipping entry 5 due to error: 'NoneType' object has no attribute 'get'
[Warning] Skipping entry 6 due to error: Expected 16384 bits, got 4096
[Warning] Skipping entry 7 due to error: Expected 16384 bits, got 4096
[Warning] Skipping entry 8 due to error: Expected 16384 bits, got 4096
[Warning] Skipping entry 9 due to error: Expected 16384 bits, got 4096
[Warning] Skipping entry 10 due to error: Expected 16384 bits, got 4096
[Warning] Skipping entry 11 due to error: Expected 16384 bits, got 4096
[Warning] Skipping entry 12 due to error: Expected 16384 bits, got 4096
[Warning

KeyboardInterrupt: 

In [ ]:
# import numpy as np
# from tqdm import tqdm

# def compute_hamming(template1, mask1, template2, mask2, shifts=range(-7, 8)):
#     """
#     Compute Hamming distance with shifting between two templates.
#     """
#     min_distance = 1.0
#     for shift in shifts:
#         t2_shifted = np.roll(template2, shift=shift, axis=2)
#         m2_shifted = np.roll(mask2, shift=shift, axis=2)

#         xor_real = template1[..., 0] ^ t2_shifted[..., 0]
#         xor_imag = template1[..., 1] ^ t2_shifted[..., 1]

#         valid_mask_real = mask1[..., 0] & m2_shifted[..., 0]
#         valid_mask_imag = mask1[..., 1] & m2_shifted[..., 1]

#         masked_xor_real = xor_real & valid_mask_real
#         masked_xor_imag = xor_imag & valid_mask_imag

#         num_diff = np.sum(masked_xor_real) + np.sum(masked_xor_imag)
#         num_valid = np.sum(valid_mask_real) + np.sum(valid_mask_imag)

#         if num_valid > 0:
#             dist = num_diff / num_valid
#             min_distance = min(min_distance, dist)

#     return min_distance

# # === Setup ===
# n = len(samples_a)
# assert len(samples_b) == n, "Samples A and B must be the same length"

# distance_matrix = np.full((n, n), fill_value=np.nan, dtype=np.float32)

# # === Compare each pair using both resolutions ===
# for i in tqdm(range(n), desc="Computing dual-resolution distances"):
#     if samples_a[i] is None or samples_b[i] is None:
#         continue

#     t1_a, m1_a = samples_a[i]
#     t1_b, m1_b = samples_b[i]

#     for j in range(i + 1, n):
#         if samples_a[j] is None or samples_b[j] is None:
#             continue

#         t2_a, m2_a = samples_a[j]
#         t2_b, m2_b = samples_b[j]

#         # Compute distances using both resolutions
#         dist_a = compute_hamming(t1_a, m1_a, t2_a, m2_a)
#         dist_b = compute_hamming(t1_b, m1_b, t2_b, m2_b)

#         # Combine them (simple average; customize if needed)
#         combined_distance = (dist_a + dist_b) / 2.0

#         distance_matrix[i, j] = combined_distance
#         distance_matrix[j, i] = combined_distance  # symmetric

# # === Save output ===
# np.save("hamming_distance_dual_resolution.npy", distance_matrix)
# print("✅ Dual-resolution distance matrix saved.")


Computing dual-resolution distances:   8%|▊         | 95/1200 [07:13<1:24:03,  4.56s/it]


KeyboardInterrupt: 

In [23]:
import numpy as np

# === Input paths ===
FILE_A = "./combined/EF-hjy_single_(16x256).npy"
FILE_B = "./combined/EF-hjy_single_(8x128).npy"
FILE_C = "./yuleQ/BMT-y_single_(8x128).npy"
OUTPUT_FILE = "./combined/weighted_avg.npy"

# === Load the matrices ===
matrix_a = np.load(FILE_A)
matrix_b = np.load(FILE_B)
matrix_c = np.load(FILE_C)

# === Check shapes match ===
if matrix_a.shape != matrix_b.shape or matrix_a.shape != matrix_c.shape or matrix_b.shape != matrix_c.shape:
    raise ValueError("Input matrices must have the same shape.")

# === Compute average (NaN-aware) ===
avg_matrix = (matrix_a + matrix_b) / 2

# === Save result ===
np.save(OUTPUT_FILE, avg_matrix)
print(f"✅ Averaged matrix saved to '{OUTPUT_FILE}'")


✅ Averaged matrix saved to './combined/weighted_avg.npy'
